<a href="https://colab.research.google.com/github/ssprajapati2021/Hybrid-RAG-Fine-Tuning/blob/main/notebook/Baseline_Model_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 3: Baseline Model Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] Internet access (to download the model)
- [ ] GPU runtime enabled (Runtime → Change runtime type → T4 GPU)

**Files this notebook will CREATE:**
- [ ] `outputs.json` — `test_query`, `ground_truth`, `baseline_output` _(Required by NB4, NB5, NB7)_

---

## **Stage 3: Solution V1 (Retrieval-Assisted Generation)**

### **Task 3.1: Establish Baseline Performance**

#### **3.1.1 Execute Baseline Inference [2 marks]**
**The Task:** Load the pre-trained base model in 4-bit quantization and generate a response to an ambiguous shipping-delay query without any context.

**Hints & Tips:**
* Use `do_sample=False` for deterministic output. Do NOT pair `temperature=0.0` with `do_sample=False` — it throws a deprecation warning. Use `temperature=None, top_p=None`.
* `BitsAndBytesConfig(load_in_4bit=True)` shrinks the 1.5B model to ~750MB VRAM.
* `max_new_tokens=120` gives room for a complete answer.

**Model Selection:**
* **Qwen/Qwen2.5-1.5B-Instruct** (recommended) — must match what you used in NB2.
* **TinyLlama-1.1B-Chat** — lighter, weaker structured output.
* **Llama-3-8B-Instruct** — best quality, may OOM on free T4 during fine-tuning.

**Learner Inference:** This establishes your zero-shot baseline. Every later improvement is measured against this exact output.

In [ ]:
# Mount Google Drive to save the outputs.json
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Installing dependencies having lower version
# !pip install -q -U bitsandbytes>=0.46.1

In [ ]:
import torch
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig)

# Configure the model which used in notebook2
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load the base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
# Ambiguous Customer Query
test_query = "My package is taking much longer than expected. Can you tell me what's happening?"

# Chat prompt
messages = [
    {
        "role": "system",
        "content": "You are a helpful customer support assistant."
    },
    {
        "role": "user",
        "content": test_query
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Tokenize the prompt
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

# Generate the Response
outputs = model.generate(
    **inputs,
    max_new_tokens=120,
    do_sample=False,
    temperature=None,
    top_p=None
)

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

assistant_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(f"Assistant:\n {assistant_response}")

Assistant:
 I'm sorry to hear that your package is taking longer than expected. There could be several reasons for this delay, such as traffic or road conditions, weather-related delays, or issues with the carrier's system. I recommend checking the status of your package on the carrier's website or by calling their customer service number. They can provide more information and help resolve any issues that may be causing the delay. If the issue persists, it might be best to contact the carrier directly to see if they can expedite delivery.


#### **3.1.2 Evaluate Baseline Quality [2 marks]**
**The Task:** Assess the baseline output for factual inaccuracies against the ground-truth SOP rule.

**Hints & Tips:**
* Compare against the known rule: "Domestic orders deliver within 3-7 business days."
* Did the model invent a timeline? Mention a non-existent tracking system or department?
* Document every hallucination — it justifies Stages 3 and 4.

**Learner Inference:** This hallucination is exactly why you build Stage 3 (a database) and Stage 4 (a router).

In [ ]:
# YOUR CODE HERE
ground_truth = "3-7 business days."
baseline_response = assistant_response.lower()

if ground_truth in baseline_response:
  print("✅ Correct delivery timeline found.")
else:
  print("❌ Delivery timeline missing or incorrect.")
  print("\n Conclusion: The baseline model did not follow the SOP and relied on general knowledge.")

❌ Delivery timeline missing or incorrect.

 Conclusion: The baseline model did not follow the SOP and relied on general knowledge.


In [ ]:
hallucinations = {
    "Invented weather delay": "weather",
    "Invented high order volume delay": "high order volumes",
    "Non-existent tracking department": "tracking department",
    "Non-existent Shipping Resolution Team": "shipping resolution team"
}

print("-------Hallucination Check--------")

# Check for unsupported claims
for description, phrase in hallucinations.items():
    if phrase in baseline_response:
        print(f"⚠️ {description}")

# Check whether the correct SOP timeline is mentioned
if ground_truth in baseline_response:
    print("✅ Correct SOP delivery timeline mentioned.")
else:
    print("❌ SOP delivery timeline (3-7 business days) missing.")

-------Hallucination Check--------
⚠️ Invented weather delay
❌ SOP delivery timeline (3-7 business days) missing.


In [ ]:
# Check for an invented timeline (optional)
import re

timeline_pattern = r"\b\d+\s*[-to]+\s*\d+\s*business days\b"

matches = re.findall(timeline_pattern, baseline_response)

for timeline in matches:
    if timeline != ground_truth:
        print(f"⚠️ Possible invented timeline: {timeline}")

---
## Save Artifacts for Downstream Notebooks

**IMPORTANT:** Saves the baseline output. Notebooks 4, 5, and 7 depend on this file.

In [ ]:
import json

outputs = {
    "test_query": test_query,
    "ground_truth": "Domestic orders deliver within 3-7 business days.",
    "baseline_output": assistant_response
}

print(outputs)

# Saving as outputs.json
artifact_path = "/content/drive/MyDrive/corporate_policies"

with open(f"{artifact_path}/outputs.json", "w") as f:
    json.dump(outputs, f, indent=4)



{'test_query': "My package is taking much longer than expected. Can you tell me what's happening?", 'ground_truth': 'Domestic orders deliver within 3-7 business days.', 'baseline_output': "I'm sorry to hear that your package is taking longer than expected. There could be several reasons for this delay, such as traffic or road conditions, weather-related delays, or issues with the carrier's system. I recommend checking the status of your package on the carrier's website or by calling their customer service number. They can provide more information and help resolve any issues that may be causing the delay. If the issue persists, it might be best to contact the carrier directly to see if they can expedite delivery."}


In [ ]:
# Verifying outputs.json
with open(f"{artifact_path}/outputs.json", "r") as f:
    print(f.read())

{
    "test_query": "My package is taking much longer than expected. Can you tell me what's happening?",
    "ground_truth": "Domestic orders deliver within 3-7 business days.",
    "baseline_output": "I'm sorry to hear that your package is taking longer than expected. There could be several reasons for this delay, such as traffic or road conditions, weather-related delays, or issues with the carrier's system. I recommend checking the status of your package on the carrier's website or by calling their customer service number. They can provide more information and help resolve any issues that may be causing the delay. If the issue persists, it might be best to contact the carrier directly to see if they can expedite delivery."
}


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 4.**

- [x] Base model loaded in 4-bit without errors
- [x] Baseline output generated for `test_query`
- [x] Hallucination assessment documented
- [x] **`outputs.json` saved** with `test_query`, `ground_truth`, `baseline_output` ← _CRITICAL for NB4, 5, 7_
- [x] GPU runtime enabled

**If any item is unchecked, fix it before moving on.**